# 🛡️ SafeGuard Vision AI - Frame Extractor
## MIT Global Teaching Labs
---
**Autor:** Christian Cajusol  
**Fecha:** Enero 2026

Este notebook extrae frames de videos para crear un dataset de entrenamiento.

### 📋 Estructura esperada:
```
Google Drive/
└── tu_carpeta/
    ├── emergencias/     ← 30 videos de 2 seg
    └── simuladas/       ← 40 videos de 11 seg
```

### 📤 Resultado:
```
Google Drive/
└── tu_carpeta/
    └── dataset_frames/
        ├── emergencias/     ← ~900 frames
        ├── simuladas/       ← ~4,400 frames
        ├── dataset_metadata.csv
        └── dataset_info.json
```

---
## 📌 Paso 1: Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive montado correctamente!")

---
## 📌 Paso 2: Configurar rutas

⚠️ **EDITA ESTAS RUTAS** según tu estructura en Drive

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    ⚠️ CONFIGURA TUS RUTAS AQUÍ ⚠️                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Carpeta base en tu Drive (donde están las subcarpetas de videos)
BASE_FOLDER = "/content/drive/MyDrive/MIT_SafeGuard"  # ← CAMBIA ESTO

# Carpetas de videos de entrada
VIDEOS_EMERGENCIAS = f"{BASE_FOLDER}/emergencias"     # ← Videos de 2 seg
VIDEOS_SIMULADAS = f"{BASE_FOLDER}/simuladas"         # ← Videos de 11 seg

# Carpeta de salida para el dataset de frames
OUTPUT_DATASET = f"{BASE_FOLDER}/dataset_frames"

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURACIÓN DE EXTRACCIÓN                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

CONFIG = {
    "emergencias": {
        "fps_extract": 15,      # Frames por segundo a extraer
        "prefix": "emerg",      # Prefijo para archivos
        "label": 1              # 1 = emergencia real
    },
    "simuladas": {
        "fps_extract": 10,      # Menos fps porque son videos más largos
        "prefix": "simul",      # Prefijo para archivos
        "label": 0              # 0 = simulada
    }
}

# Formato de imagen
IMAGE_FORMAT = "jpg"
JPEG_QUALITY = 95

print("📁 Configuración de rutas:")
print(f"   Videos emergencias: {VIDEOS_EMERGENCIAS}")
print(f"   Videos simuladas:   {VIDEOS_SIMULADAS}")
print(f"   Dataset salida:     {OUTPUT_DATASET}")

---
## 📌 Paso 3: (Opcional) Explorar tu Drive

Ejecuta esto si no estás seguro de la ruta exacta

In [ ]:
import os

# Cambia esta ruta para explorar diferentes carpetas
EXPLORAR = "/content/drive/MyDrive"

print(f"📂 Contenido de: {EXPLORAR}\n")
try:
    items = sorted(os.listdir(EXPLORAR))
    for item in items[:30]:  # Mostrar máximo 30 items
        full_path = os.path.join(EXPLORAR, item)
        if os.path.isdir(full_path):
            print(f"   📁 {item}/")
        else:
            print(f"   📄 {item}")
    if len(items) > 30:
        print(f"   ... y {len(items) - 30} más")
except FileNotFoundError:
    print(f"❌ No existe: {EXPLORAR}")

---
## 📌 Paso 4: Verificar carpetas y videos

In [ ]:
import cv2
import os

def get_video_info(video_path):
    """Obtiene información del video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    info = {
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "total_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "duration_sec": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / max(cap.get(cv2.CAP_PROP_FPS), 1)
    }
    cap.release()
    return info

def list_videos(folder):
    """Lista videos en una carpeta."""
    if not os.path.exists(folder):
        return []
    extensions = ('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV')
    return sorted([f for f in os.listdir(folder) if f.endswith(extensions)])

print("="*60)
print("📹 VERIFICACIÓN DE VIDEOS")
print("="*60)

total_frames_estimado = 0

for name, folder, config_key in [("EMERGENCIAS", VIDEOS_EMERGENCIAS, "emergencias"),
                                   ("SIMULADAS", VIDEOS_SIMULADAS, "simuladas")]:
    print(f"\n📂 {name}")
    print(f"   Ruta: {folder}")

    if not os.path.exists(folder):
        print(f"   ❌ ERROR: La carpeta no existe!")
        continue

    videos = list_videos(folder)
    print(f"   ✅ Videos encontrados: {len(videos)}")

    if videos:
        # Info del primer video
        info = get_video_info(os.path.join(folder, videos[0]))
        if info:
            print(f"   📏 Resolución: {info['width']}x{info['height']}")
            print(f"   🎬 FPS original: {info['fps']:.1f}")
            print(f"   ⏱️  Duración ejemplo: {info['duration_sec']:.1f} seg")

            # Estimar frames a extraer
            fps_extract = CONFIG[config_key]["fps_extract"]
            frames_por_video = int(info['duration_sec'] * fps_extract)
            total_categoria = frames_por_video * len(videos)
            total_frames_estimado += total_categoria

            print(f"   📊 Frames estimados: ~{frames_por_video} por video × {len(videos)} = ~{total_categoria}")

print("\n" + "="*60)
print(f"📊 TOTAL ESTIMADO: ~{total_frames_estimado} frames")
print("="*60)

---
## 📌 Paso 5: Cargar funciones de extracción

In [ ]:
import cv2
import os
import json
from datetime import datetime
from tqdm.notebook import tqdm

def extract_frames_from_video(video_path, output_folder, prefix, fps_extract, label, video_num):
    """
    Extrae frames de un video a una tasa específica.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"error": f"No se pudo abrir: {video_path}"}

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calcular intervalo de extracción
    frame_interval = max(1, int(video_fps / fps_extract))

    frames_extracted = 0
    frame_count = 0
    frame_data = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            frames_extracted += 1

            filename = f"{prefix}_v{video_num:03d}_f{frames_extracted:04d}.{IMAGE_FORMAT}"
            filepath = os.path.join(output_folder, filename)

            if IMAGE_FORMAT == "jpg":
                cv2.imwrite(filepath, frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            else:
                cv2.imwrite(filepath, frame)

            frame_data.append({
                "filename": filename,
                "video_source": os.path.basename(video_path),
                "frame_number": frame_count,
                "label": label,
                "label_name": "emergencia" if label == 1 else "simulada",
                "timestamp_sec": frame_count / video_fps if video_fps > 0 else 0
            })

        frame_count += 1

    cap.release()

    return {
        "video": os.path.basename(video_path),
        "total_frames_video": total_frames,
        "frames_extracted": frames_extracted,
        "frame_data": frame_data
    }


def process_category(videos_folder, output_folder, config, category_name):
    """Procesa todos los videos de una categoría."""

    os.makedirs(output_folder, exist_ok=True)

    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV')
    videos = sorted([f for f in os.listdir(videos_folder) if f.endswith(video_extensions)])

    if not videos:
        print(f"⚠️ No se encontraron videos en: {videos_folder}")
        return None

    print(f"\n{'='*60}")
    print(f"📂 Procesando: {category_name.upper()}")
    print(f"{'='*60}")
    print(f"   📁 Entrada:  {videos_folder}")
    print(f"   📁 Salida:   {output_folder}")
    print(f"   🎬 Videos:   {len(videos)}")
    print(f"   ⚡ FPS ext:  {config['fps_extract']} fps")

    all_frame_data = []
    total_frames = 0

    for i, video_name in enumerate(tqdm(videos, desc=f"Extrayendo {category_name}")):
        video_path = os.path.join(videos_folder, video_name)

        result = extract_frames_from_video(
            video_path=video_path,
            output_folder=output_folder,
            prefix=config["prefix"],
            fps_extract=config["fps_extract"],
            label=config["label"],
            video_num=i + 1
        )

        if "error" not in result:
            total_frames += result["frames_extracted"]
            all_frame_data.extend(result["frame_data"])

    print(f"   ✅ Completado: {total_frames} frames extraídos")

    return {
        "category": category_name,
        "videos_processed": len(videos),
        "total_frames": total_frames,
        "frame_data": all_frame_data
    }


def save_metadata(all_data, output_folder):
    """Guarda metadata en CSV y JSON."""

    all_frames = []
    for category_data in all_data:
        if category_data:
            all_frames.extend(category_data["frame_data"])

    # CSV
    csv_path = os.path.join(output_folder, "dataset_metadata.csv")
    with open(csv_path, 'w', encoding='utf-8') as f:
        f.write("filename,video_source,frame_number,label,label_name,timestamp_sec\n")
        for frame in all_frames:
            f.write(f"{frame['filename']},{frame['video_source']},{frame['frame_number']},"
                   f"{frame['label']},{frame['label_name']},{frame['timestamp_sec']:.3f}\n")

    # JSON
    json_path = os.path.join(output_folder, "dataset_info.json")
    summary = {
        "created": datetime.now().isoformat(),
        "project": "SafeGuard Vision AI",
        "author": "Christian Cajusol - MIT Global Teaching Labs",
        "image_size": "320x240",
        "format": IMAGE_FORMAT,
        "categories": {
            "emergencias": {"label": 1, "description": "Caídas reales/emergencias"},
            "simuladas": {"label": 0, "description": "Caídas actuadas/simuladas"}
        },
        "statistics": {
            cat["category"]: {
                "videos": cat["videos_processed"],
                "frames": cat["total_frames"]
            } for cat in all_data if cat
        },
        "total_frames": sum(cat["total_frames"] for cat in all_data if cat)
    }

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    return csv_path, json_path

print("✅ Funciones cargadas correctamente!")

---
## 📌 Paso 6: 🚀 EXTRAER TODOS LOS FRAMES

⚠️ Esto puede tomar varios minutos dependiendo de la cantidad de videos

In [ ]:
print("\n")
print("╔" + "═"*58 + "╗")
print("║" + " 🛡️  SafeGuard Vision AI - Frame Extractor ".center(58) + "║")
print("║" + " MIT Global Teaching Labs ".center(58) + "║")
print("╚" + "═"*58 + "╝")

# Crear carpeta de salida principal
os.makedirs(OUTPUT_DATASET, exist_ok=True)

all_results = []

# Procesar EMERGENCIAS
if os.path.exists(VIDEOS_EMERGENCIAS):
    result_emerg = process_category(
        videos_folder=VIDEOS_EMERGENCIAS,
        output_folder=os.path.join(OUTPUT_DATASET, "emergencias"),
        config=CONFIG["emergencias"],
        category_name="emergencias"
    )
    all_results.append(result_emerg)
else:
    print(f"\n❌ No existe: {VIDEOS_EMERGENCIAS}")

# Procesar SIMULADAS
if os.path.exists(VIDEOS_SIMULADAS):
    result_simul = process_category(
        videos_folder=VIDEOS_SIMULADAS,
        output_folder=os.path.join(OUTPUT_DATASET, "simuladas"),
        config=CONFIG["simuladas"],
        category_name="simuladas"
    )
    all_results.append(result_simul)
else:
    print(f"\n❌ No existe: {VIDEOS_SIMULADAS}")

# Guardar metadata
if any(all_results):
    csv_path, json_path = save_metadata(all_results, OUTPUT_DATASET)

    # Resumen final
    total_frames = sum(cat["total_frames"] for cat in all_results if cat)

    print("\n")
    print("╔" + "═"*58 + "╗")
    print("║" + " 🎉 EXTRACCIÓN COMPLETADA ".center(58) + "║")
    print("╠" + "═"*58 + "╣")

    for cat in all_results:
        if cat:
            line = f"  📂 {cat['category'].upper():<15} {cat['total_frames']:>6} frames  ({cat['videos_processed']} videos)"
            print(f"║{line:<58}║")

    print("╠" + "═"*58 + "╣")
    total_line = f"  📊 TOTAL DATASET: {total_frames:>10} frames"
    print(f"║{total_line:<58}║")
    print("╠" + "═"*58 + "╣")
    print(f"║  📄 CSV:  dataset_metadata.csv{' '*25}║")
    print(f"║  📄 JSON: dataset_info.json{' '*28}║")
    print(f"║  📂 Ruta: {OUTPUT_DATASET[-45:]:<47}║")
    print("╚" + "═"*58 + "╝")
else:
    print("\n❌ No se procesó ningún video. Verifica las rutas.")

---
## 📌 Paso 7: Verificar resultados

In [ ]:
import os

print("📂 Contenido del dataset generado:\n")

for category in ["emergencias", "simuladas"]:
    folder = os.path.join(OUTPUT_DATASET, category)
    if os.path.exists(folder):
        files = [f for f in os.listdir(folder) if f.endswith(('.jpg', '.png'))]
        print(f"   📁 {category}/")
        print(f"      └─ {len(files)} imágenes")
        if files:
            print(f"      └─ Ejemplo: {files[0]}")

# Mostrar archivos de metadata
print(f"\n   📄 dataset_metadata.csv")
print(f"   📄 dataset_info.json")

print(f"\n📂 Ubicación: {OUTPUT_DATASET}")

---
## 📌 Paso 8: (Opcional) Ver muestra de imágenes

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('🛡️ SafeGuard Vision AI - Muestra del Dataset', fontsize=14, fontweight='bold')

for row, category in enumerate(["emergencias", "simuladas"]):
    folder = os.path.join(OUTPUT_DATASET, category)
    if os.path.exists(folder):
        images = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png'))])

        # Seleccionar 4 imágenes espaciadas
        if len(images) >= 4:
            step = len(images) // 4
            selected = [images[i * step] for i in range(4)]
        else:
            selected = images[:4]

        for col, img_name in enumerate(selected):
            img_path = os.path.join(folder, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            axes[row, col].imshow(img)
            axes[row, col].set_title(f"{category}\n{img_name}", fontsize=8)
            axes[row, col].axis('off')

plt.tight_layout()
plt.show()

---
## 📌 Paso 9: (Opcional) Ver CSV de metadata

In [ ]:
import pandas as pd

csv_path = os.path.join(OUTPUT_DATASET, "dataset_metadata.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

    print("📊 RESUMEN DEL DATASET")
    print("="*50)
    print(f"\nTotal de frames: {len(df)}")
    print(f"\nDistribución por clase:")
    print(df['label_name'].value_counts())

    print(f"\n\n📋 Primeras 10 filas:")
    display(df.head(10))

    print(f"\n📋 Últimas 10 filas:")
    display(df.tail(10))
else:
    print("❌ No se encontró el archivo CSV")

---
## ✅ ¡Listo!

Tu dataset está en Google Drive en la carpeta especificada.

### 📁 Estructura final:
```
dataset_frames/
├── emergencias/
│   ├── emerg_v001_f0001.jpg
│   ├── emerg_v001_f0002.jpg
│   └── ...
├── simuladas/
│   ├── simul_v001_f0001.jpg
│   └── ...
├── dataset_metadata.csv
└── dataset_info.json
```

### 🚀 Próximos pasos:
1. **BlazePose**: Extraer keypoints de cada frame
2. **Entrenar clasificador**: emergencia vs simulada
3. **Evaluar modelo**: métricas de detección

---
*SafeGuard Vision AI - MIT Global Teaching Labs*